In [40]:
import numpy as np
import polars as pl
from rapidfuzz import fuzz as rapidfuzz_fuzz, process

In [41]:
textos = (
    pl.read_parquet('../database/textos/*.parquet')
    .join(
        pl.read_parquet('../database/metadata/*.parquet'),
        on='id'
    )
    .with_columns(
        pl.col('pubDate').str.to_date(format='%d/%m/%Y')
    )
    .filter(
        pl.col('pubDate').dt.year()>=2018,
        pl.col('artCategory').str.to_lowercase().str.contains(r'(marinha|ex[ée]rcito|aeron[áa]utica)')
    )
).select('id', 'texto')

In [42]:
textos

id,texto
str,str
"""515_20190102_11355821""",""" PORTARIA Nº 2.082, DE 27 DE D…"
"""515_20190102_11355822""",""" PORTARIA Nº 2.083, DE 27 DE D…"
"""515_20190104_11366759""",""" PORTARIA Nº 401/DPC, DE 19 DE…"
"""515_20190107_11365668-1""",""" PORTARIA DECEA Nº 258/JJAER,…"
"""515_20190107_11365668-2""",""" Art. 38 As sessões serão públ…"
…,…
"""530_20251231_23469728""","""EXTRATO DE CREDENCIAMENTO Nº 3…"
"""530_20251231_23470217""","""EXTRATO DE CREDENCIAMENTO Nº 3…"
"""530_20251231_23470438""","""EXTRATO DE CREDENCIAMENTO Nº 2…"


In [43]:
VALOR_PATTERN = r'R\$\s?(?:\d{1,3}(?:\.\d{3})*|\d+)(?:,\d{2})?'
ENT_PATTERN = r'[A-ZÁÂÃÉÊÍÓÔÕÚÇ]+\s[A-ZÁÂÃÉÊÍÓÔÕÚÇ\s]+'
PROCESSO_PATTERN = r'\b\d{5}\.?\d{6}\/?\d{4}\-?\d{2}\b'
CNPJ_PATTERN = r'\b\d{2}\.?\d{3}\.?\d{3}\/?\d{4}\-?\d{2}\b'
DATA_PATTERN = r'\b\d{1,2}/\d{1,2}/\d{2,4}\b|\b\d{1,2}\s+de\s+[a-zçãõáéíóúâêô]+\s+de\s+\d{4}\b'

In [44]:
def filter_nested_entities(items):
    if items is None:
        return []

    normalized_items = [item.strip() for item in items if item and item.strip()]

    return [
        item
        for item in normalized_items
        if not any(item != other and item in other for other in normalized_items)
    ]

textos = textos.with_columns(
    valores=pl.col('texto').str.extract_all(VALOR_PATTERN),
    entidades=pl.col('texto').str.extract_all(ENT_PATTERN).map_elements(
        filter_nested_entities,
        return_dtype=pl.List(pl.String)
    ),
    nups=pl.col('texto').str.extract_all(PROCESSO_PATTERN),
    cnpjs=pl.col('texto').str.extract_all(CNPJ_PATTERN),
    datas=pl.col('texto').str.to_lowercase().str.extract_all(DATA_PATTERN)
)

In [45]:
textos

id,texto,valores,entidades,nups,cnpjs,datas
str,str,list[str],list[str],list[str],list[str],list[str]
"""515_20190102_11355821""",""" PORTARIA Nº 2.082, DE 27 DE D…",[],"[""PORTARIA N"", ""DE DEZEMBRO DE"", … ""EDUARDO DIAS DA COSTA VILLAS BÔAS""]",[],[],"[""27 de dezembro de 2018"", ""30 de novembro de 2010"", … ""30 de novembro de 2010""]"
"""515_20190102_11355822""",""" PORTARIA Nº 2.083, DE 27 DE D…",[],"[""PORTARIA N"", ""DE DEZEMBRO DE"", … ""EDUARDO DIAS DA COSTA VILLAS BÔAS""]",[],[],"[""27 de dezembro de 2018"", ""30 de novembro de 2010"", … ""30 de novembro de 2010""]"
"""515_20190104_11366759""",""" PORTARIA Nº 401/DPC, DE 19 DE…",[],"[""PORTARIA N"", ""DE DEZEMBRO DE"", … ""ROBERTO GONDIM CARNEIRO DA CUNHA""]",[],[],"[""19 de dezembro de 2018"", ""3 de junho de 2004"", … ""29 de novembro de 2018""]"
"""515_20190107_11365668-1""",""" PORTARIA DECEA Nº 258/JJAER,…",[],"[""PORTARIA DECEA N"", ""DE DEZEMBRO DE"", … ""O P""]",[],[],"[""10 de dezembro de 2018"", ""16 de setembro de 2013"", … ""19 de dezembro de 1986""]"
"""515_20190107_11365668-2""",""" Art. 38 As sessões serão públ…",[],"[""V D"", ""A J"", … ""TRANSMISSÃO ELETRÔNICA""]",[],[],[]
…,…,…,…,…,…,…
"""530_20251231_23469728""","""EXTRATO DE CREDENCIAMENTO Nº 3…","[""R$ 151.062,00""]","[""EXTRATO DE CREDENCIAMENTO N"", ""HOSPITAL NAVAL DE NATAL"", … ""NÃO SE APLICA""]","[""63064.010971/2023-38""]","[""08.242.471/0001-18""]","[""15/12/2025"", ""14/12/2026"", … ""30/12/2025""]"
"""530_20251231_23470217""","""EXTRATO DE CREDENCIAMENTO Nº 3…","[""R$ 20.000,00""]","[""EXTRATO DE CREDENCIAMENTO N"", ""HOSPITAL NAVAL DE NATAL"", … ""NÃO SE APLICA""]","[""63064.010971/2023-38""]","[""60.143.972/0001-67""]","[""15/12/2025"", ""14/12/2026"", … ""30/12/2025""]"
"""530_20251231_23470438""","""EXTRATO DE CREDENCIAMENTO Nº 2…","[""R$ 20.000,00""]","[""EXTRATO DE CREDENCIAMENTO N"", ""HOSPITAL NAVAL DE NATAL"", … ""NÃO SE APLICA""]","[""63064.010971/2023-38""]","[""70.030.606/0001-55""]","[""09/12/2025"", ""08/12/2026"", … ""30/12/2025""]"


In [46]:
print(textos.filter(pl.col('valores').list.len() > 0).item(6, 'texto'))

PORTARIA Nº 184/HNRE, DE 18 DE DEZEMBRO DE 2020
O DIRETOR DO HOSPITAL NAVAL DE RECIFE, em conformidade com contido na Orientação Normativa AGU nº 33/2011, resolve:
Art. 1º Que seja dada publicidade, por meio do Diário Oficial da União, aos Termos de Adesão ao Edital de Credenciamento nº 2/2019, Processo Administrativo n° 63066.003213/2019-67, deste Hospital, assinado pela Organização de Saúde Extra-Marinha abaixo especificada:
I - COOPERATIVA DOS MÉDICOS ANESTESIOLOGISTAS DE PERNAMBUCO.
a)CNPJ nº 11.187.085/0001-85; e
b) Valor Estimativo: R$ 200.000,00 (duzentos mil reais).
Art. 2º Fundamentação: Inexigibilidade de Licitação, com fundamento no art. 25, caput, da Lei nº 8.666/1993.
Art. 3º Esta Portaria entra em vigor na presente data.
Capitão de Mar e Guerra (Md) CÁSSIO DE SOUZA SANTOS



In [47]:
print(textos.item(1, 'texto'))


PORTARIA Nº 2.083, DE 27 DE DEZEMBRO DE 2018
Fixa as metas de desempenho institucional para o ano de 2019, no âmbito do Exército, para fim de aplicação da Portaria do Comandante do Exército nº 1.180, de 30 de novembro de 2010.
O COMANDANTE DO EXÉRCITO, no uso das atribuições que lhe conferem o art. 4º da Lei Complementar nº 97, de 9 de junho de 1999, alterada pela Lei Complementar nº 136, de 25 de agosto de 2010, o inciso I do art. 20 da Estrutura Regimental do Comando do Exército, aprovada pelo Decreto nº 5.751, de 12 de abril de 2006, e em conformidade com o Decreto nº 7.133, de 19 de março de 2010, o parágrafo 1º do art. 21 da Portaria do Comandante do Exército nº 1.180, de 30 de novembro de 2010, ouvido o Estado-Maior do Exército, resolve:
Art. 1º Fixar as metas globais de desempenho institucional para o ano de 2019, no âmbito do Exército, para fim de aplicação da Portaria do Comandante do Exército nº 1.180, de 30 de novembro de 2010.
Art. 2º Determinar que o Centro de Comunicação

In [48]:
textos.item(1,  'entidades')

""
str
"""PORTARIA N"""
"""DE DEZEMBRO DE"""
"""O COMANDANTE DO EXÉRCITO"""
"""METAS GLOBAIS DE DESEMPENHO IN…"
"""OBJETIVO ESTRATÉGICO RESPONSÁV…"
"""EDUARDO DIAS DA COSTA VILLAS B…"


In [49]:
contagem_entidades = {}

for i in range(textos.height):
    for entidade in textos.item(i, 'entidades'):
        if entidade not in contagem_entidades:
            contagem_entidades[entidade] = 1
        else:
            contagem_entidades[entidade] += 1

In [50]:
contagem_entidades

{'PORTARIA N': 65315,
 'DE DEZEMBRO DE': 7244,
 'O COMANDANTE DO EXÉRCITO': 6071,
 'RESULTADO DO DESEMPENHO INSTITUCIONAL DO EB': 1,
 'OBJETIVO ESTRATÉGICO\nRESPONSÁVEL\nINDICADOR\nFÓRMULA\nMETA\nDESEMPENHO\nF': 2,
 'DESEMPENHO GLOBAL': 7,
 'EDUARDO DIAS DA COSTA VILLAS BÔAS': 710,
 'METAS GLOBAIS DE DESEMPENHO INSTITUCIONAL PARA O ANO DE': 5,
 'OBJETIVO ESTRATÉGICO\nRESPONSÁVEL PELO INDICADOR\nINDICADOR\nFÓRMULA\nMETA\nF': 1,
 'O DIRETOR DE PORTOS E COSTAS': 1519,
 'CONSIDERAÇÕES GERAIS': 22,
 'ÁREAS DE SEGURANÇA': 10,
 'ROBERTO GONDIM CARNEIRO DA CUNHA': 435,
 'PORTARIA DECEA N': 113,
 'O DIRETOR': 523,
 'GERAL DO DEPARTAMENTO DE CONTROLE DO ESPAÇO AÉREO': 127,
 'JEFERSON DOMINGUES DE FREITAS\n\nANEXO I\nREGULAMENTO DA COMPETÊNCIA': 1,
 'FUNCIONAMENTO E PROCEDIMENTO DOS PROCESSOS DA JUNTA DE JULGAMENTO DA AERONÁUTICA': 1,
 'RJJAER\nTÍTULO I\nDA COMPETÊNCIA': 1,
 'ORGANIZAÇÃO E FUNCIONAMENTO DA JUNTA DE JULGAMENTO DA AERONÁUTICA\nCAPÍTULO I\nDA FINALIDADE': 1,
 'FUNCIONAMENTO E JURISD

In [51]:
contagem = pl.DataFrame(
    {
        'entidades': list(contagem_entidades.keys()),
        'contagem': list(contagem_entidades.values())
    }
).sort('contagem', descending=True)

In [52]:
contagem

entidades,contagem
str,i64
"""CPF CONTRATADA""",97409
"""EXTRATO DE TERMO ADITIVO N""",70990
"""PORTARIA N""",65315
"""AVISO DE LICITAÇÃO P""",64528
"""EXTRATO DE CONTRATO N""",61990
…,…
"""SECRETARIA DE ESTADO DE EDUCAC…",1
"""CENTRO DE REABILITACAO E SAUDE…",1
"""CIM CARE""",1


In [53]:
covered_entities = set()
maybe_matches = {}
ents = contagem.filter(pl.col('contagem') > 5).get_column('entidades').to_list()
ents.sort(key=len)
ents = ents[::-1]
total = len(ents)

for i, ent in enumerate(ents[:-1]):
    print(f'{i*100/total:.2f}%', end='\r')
    if ent in covered_entities:
        continue

    tail = [candidate for candidate in ents[i + 1:] if candidate not in covered_entities]
    if not tail:
        continue

    scores = process.cdist(
        [ent],
        tail,
        scorer=rapidfuzz_fuzz.ratio,
        score_cutoff=81,
        dtype=np.uint8,
        workers=-1,
    )[0]
    matches = [candidate for candidate, score in zip(tail, scores) if score > 80]
    if matches:
        maybe_matches[ent] = matches
        covered_entities.update(matches)
        
maybe_matches = {k:list(set(v)) for k, v in maybe_matches.items()}

In [54]:
maybe_matches

{'PROGRAMA E BIBLIOGRAFIA PARA A PROVA ESCRITA DE CONHECIMENTOS PROFISSIONAIS PARA O CONCURSO PÚBLICO DE ADMISSÃO AO CURSO DE FORMAÇÃO PARA O INGRESSO NO CORPO AUXILIAR DE PRAÇAS DA MARINHA': ['DE CONHECIMENTOS PROFISSIONAIS PARA O CONCURSO PÚBLICO DE ADMISSÃO AO CURSO DE FORMAÇÃO PARA O INGRESSO NO CORPO AUXILIAR DE PRAÇAS DA MARINHA'],
 'DE CONHECIMENTOS PROFISSIONAIS\nPROGRAMA E BIBLIOGRAFIA PARA A PROVA ESCRITA DE CONHECIMENTOS PROFISSIONAIS PARA INGRESSO NO QUADRO COMPLEMENTAR DO CORPO DE INTENDENTES DA MARINHA': ['PROGRAMA E BIBLIOGRAFIA PARA A PROVA ESCRITA DE CONHECIMENTOS PROFISSIONAIS PARA INGRESSO NO QUADRO COMPLEMENTAR DO CORPO DE INTENDENTES DA MARINHA'],
 'MINISTÉRIO DA DEFESA\nCOMANDO DA AERONÁUTICA\nESCOLA DE ESPECIALISTAS DE AERONÁUTICA\nAUTORIZAÇÃO PARA CANDIDATO MENOR QUE OPTOU PELO SISTEMA DE RESERVA DE VAGAS': ['MINISTÉRIO DA DEFESA\nCOMANDO DA AERONÁUTICA\nESCOLA DE ESPECIALISTAS DE AERONÁUTICA\nAUTORIZAÇÃO PARA CANDIDATO MENOR DE IDADE\nE'],
 'EXTRATO DE TERMO AD

In [55]:
def find_substring(s_key, s_list):
    s_list.append(s_key)
    for s in set(s_list):
        align = rapidfuzz_fuzz.partial_ratio_alignment(min(s_key, s, key=len), max(s_key, s, key=len))
        s_key = min(s_key, s, key=len)[align.dest_start: align.dest_end]
    return {s_key:list(set(s_list))}

In [56]:
lean_matches = {}
for item in maybe_matches.items():
    lean_matches.update(find_substring(*item))


In [57]:
lean_matches

{'MARINHA': ['CAIXA DE CONSTRUÇÕES DE CASAS PARA O PESSOAL DA MARINHA',
  'O PRESIDENTE DA CAIXA DE CONSTRUÇÕES DE CASAS PARA O PESSOAL DA MARINHA'],
 'PROVA ESCRITA DE CONHECIMENTOS PROFISSIONAIS PARA INGRESSO NO QUADRO COMPLEMENTAR DO CORPO DE INTENDENTES DA MARINHA': ['DE CONHECIMENTOS PROFISSIONAIS\nPROGRAMA E BIBLIOGRAFIA PARA A PROVA ESCRITA DE CONHECIMENTOS PROFISSIONAIS PARA INGRESSO NO QUADRO COMPLEMENTAR DO CORPO DE INTENDENTES DA MARINHA',
  'PROGRAMA E BIBLIOGRAFIA PARA A PROVA ESCRITA DE CONHECIMENTOS PROFISSIONAIS PARA INGRESSO NO QUADRO COMPLEMENTAR DO CORPO DE INTENDENTES DA MARINHA'],
 'MINISTÉRIO DA DEFESA\nCOMANDO DA AERONÁUTICA\nESCOLA DE ESPECIALISTAS DE AERONÁUTICA\nAUTORIZAÇÃO PARA CANDIDATO MENOR DE I': ['MINISTÉRIO DA DEFESA\nCOMANDO DA AERONÁUTICA\nESCOLA DE ESPECIALISTAS DE AERONÁUTICA\nAUTORIZAÇÃO PARA CANDIDATO MENOR QUE OPTOU PELO SISTEMA DE RESERVA DE VAGAS',
  'MINISTÉRIO DA DEFESA\nCOMANDO DA AERONÁUTICA\nESCOLA DE ESPECIALISTAS DE AERONÁUTICA\nAUTORIZA

In [58]:
test_key = list(maybe_matches.keys())[0]
test_value = maybe_matches[test_key][0]

print(test_key, test_value, sep='\n')

PROGRAMA E BIBLIOGRAFIA PARA A PROVA ESCRITA DE CONHECIMENTOS PROFISSIONAIS PARA O CONCURSO PÚBLICO DE ADMISSÃO AO CURSO DE FORMAÇÃO PARA O INGRESSO NO CORPO AUXILIAR DE PRAÇAS DA MARINHA
DE CONHECIMENTOS PROFISSIONAIS PARA O CONCURSO PÚBLICO DE ADMISSÃO AO CURSO DE FORMAÇÃO PARA O INGRESSO NO CORPO AUXILIAR DE PRAÇAS DA MARINHA


In [59]:
align = rapidfuzz_fuzz.partial_ratio_alignment(min(test_key, test_value, key=len), max(test_key, test_value, key=len))

In [60]:
min(test_key, test_value, key=len)[align.dest_start:align.dest_end]

'O PÚBLICO DE ADMISSÃO AO CURSO DE FORMAÇÃO PARA O INGRESSO NO CORPO AUXILIAR DE PRAÇAS DA MARINHA'

In [61]:
min(test_key, test_value, key=len)

'DE CONHECIMENTOS PROFISSIONAIS PARA O CONCURSO PÚBLICO DE ADMISSÃO AO CURSO DE FORMAÇÃO PARA O INGRESSO NO CORPO AUXILIAR DE PRAÇAS DA MARINHA'

In [62]:
find_substring(test_key, maybe_matches[test_key])

{'MARINHA': ['DE CONHECIMENTOS PROFISSIONAIS PARA O CONCURSO PÚBLICO DE ADMISSÃO AO CURSO DE FORMAÇÃO PARA O INGRESSO NO CORPO AUXILIAR DE PRAÇAS DA MARINHA',
  'PROGRAMA E BIBLIOGRAFIA PARA A PROVA ESCRITA DE CONHECIMENTOS PROFISSIONAIS PARA O CONCURSO PÚBLICO DE ADMISSÃO AO CURSO DE FORMAÇÃO PARA O INGRESSO NO CORPO AUXILIAR DE PRAÇAS DA MARINHA']}

In [63]:
all_keys = list(maybe_matches.keys())

for key in all_keys:
    

_IncompleteInputError: incomplete input (1631670437.py, line 4)